# Follow-up Experiments — Kaggle 2×T4 Runner

Open this notebook in the Kaggle session with both T4 GPUs enabled and run the cells in order.
This notebook launches the scripts in the Lokesh/followup_experiments bundle from the current kernel.

Before running, make sure the code tree is available at a writable path in /kaggle/working and set the
dataset/model mount paths in the setup cell. Experiment 02 should use the full PanoX list so that all
environment folds are represented. Experiment 03 is the optional two-GPU student-training stage.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
import torch

print('CUDA available:', torch.cuda.is_available())
print('CUDA devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
subprocess.run(['nvidia-smi'], check=False)

# Set this explicitly if automatic discovery does not find your writable Lokesh checkout.
candidate_roots = [
    Path(os.environ.get('LOKESH_ROOT', '/kaggle/working/Distortion-Aware-Panoramic-Depth-Estimation/Lokesh')),
    Path('/kaggle/working/Lokesh'),
    Path('/kaggle/working/Distortion-Aware-Panoramic-Depth-Estimation/Lokesh'),
]
LOKESH_ROOT = next((p.resolve() for p in candidate_roots if (p / 'run_experiment.py').exists()
                    and (p / 'followup_experiments' / 'run_all.sh').exists()), None)
if LOKESH_ROOT is None:
    raise FileNotFoundError('Could not find a writable Lokesh checkout. Set LOKESH_ROOT to its path in /kaggle/working.')
os.chdir(LOKESH_ROOT)
sys.path.insert(0, str(LOKESH_ROOT))
print('Using Lokesh project:', LOKESH_ROOT)


In [ ]:
# 1) Choose the experiment before running this setup cell.
# Start with RUN='01' + LIMIT=16. For Experiment 02, LIMIT must remain None.
RUN = '01'                 # '01', '02', '03', '04', '05', or 'all'
LIMIT = 16
RUN_FINETUNE_IN_ALL = False

# 2) Set these to the Kaggle Input dataset folders. Each may contain extracted
#    files directly OR a .zip archive; this cell can extract archives to /kaggle/working.
DATA_PATHS = {
    'PANOX_ROOT': '/kaggle/input/panox',
    'STRUCTURED3D_ROOT': '/kaggle/input/structured3d',
    'DA360_REPO': '/kaggle/working/DA360',
    'DA360_WEIGHTS': '/kaggle/working/da360_weights/DA360_large.pth',
    'MIDAS_WEIGHTS': '/kaggle/working/midas_weights/dpt_hybrid_384.pt',
}
os.environ.update({k: str(v) for k, v in DATA_PATHS.items()})
os.environ['PYTHONUNBUFFERED'] = '1'

# Locate the repository checkout in /kaggle/working.
candidate_roots = [
    Path(os.environ.get('LOKESH_ROOT', '/kaggle/working/Distortion-Aware-Panoramic-Depth-Estimation/Lokesh')),
    Path('/kaggle/working/Lokesh'),
    Path('/kaggle/working/Distortion-Aware-Panoramic-Depth-Estimation/Lokesh'),
]
LOKESH_ROOT = next((p.resolve() for p in candidate_roots
                    if (p / 'run_experiment.py').exists()
                    and (p / 'followup_experiments' / 'run_all.sh').exists()), None)
if LOKESH_ROOT is None:
    raise FileNotFoundError(
        'Clone the committed repository into /kaggle/working first; see the README setup steps.'
    )
os.chdir(LOKESH_ROOT)
sys.path.insert(0, str(LOKESH_ROOT))
os.environ['LOKESH_ROOT'] = str(LOKESH_ROOT)
print('Lokesh root:', LOKESH_ROOT)

# Kaggle normally has CUDA-enabled PyTorch already. Do NOT pip-install torch here.
if not torch.cuda.is_available():
    raise RuntimeError('Enable Settings > Accelerator > GPU T4 x2 before continuing.')
print('CUDA devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
subprocess.run(['nvidia-smi'], check=False)
if RUN == '03' and torch.cuda.device_count() < 2:
    raise RuntimeError('Experiment 03 training is configured for both T4 GPUs.')

# Install only helper libraries; leave Kaggle's CUDA torch/torchvision untouched.
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'timm==0.6.13', 'einops', 'huggingface_hub', 'OpenEXR>=3.2'
])

# Resolve data roots. If the Kaggle Input contains the original zip file, extract
# it into writable /kaggle/working and discover the folder containing test.txt.
def resolve_dataset_root(env_key):
    root = Path(os.environ[env_key])
    if root.is_file() and root.suffix.lower() == '.zip':
        archives = [root]
        search_root = root.parent
    else:
        search_root = root
        archives = list(root.rglob('*.zip')) if root.exists() else []
    candidates = list(search_root.rglob('test.txt')) if search_root.exists() else []
    if not candidates and archives:
        for archive in archives:
            destination = Path('/kaggle/working/followup_data') / env_key.lower()
            destination.mkdir(parents=True, exist_ok=True)
            print('Extracting', archive, 'to', destination)
            import zipfile
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(destination)
        candidates = list(Path('/kaggle/working/followup_data').rglob('test.txt'))
    if root.exists() and (root / 'test.txt').is_file():
        resolved = root
    elif len(candidates) == 1:
        resolved = candidates[0].parent
    elif len(candidates) > 1:
        matching = [p.parent for p in candidates if p.parent.name.lower() in
                    (env_key.replace('_ROOT', '').lower(), 'data')]
        if len(matching) == 1:
            resolved = matching[0]
        else:
            raise RuntimeError(f'Multiple test.txt files found for {env_key}: {candidates}. Set the root to the exact folder.')
    else:
        raise FileNotFoundError(
            f'No test.txt found for {env_key}. Add the dataset in the Kaggle Input pane or correct DATA_PATHS.'
        )
    os.environ[env_key] = str(resolved.resolve())
    print(env_key, '=>', resolved, '| test.txt found')
    return resolved

needs_da360 = RUN in ('01', '02', '03', 'all')
needs_midas = RUN in ('04', '05', 'all')
needs_panox = RUN in ('02', '03', '04', '05', 'all')
needs_structured = RUN in ('01', 'all')
if needs_panox:
    panox_root = resolve_dataset_root('PANOX_ROOT')
if needs_structured:
    structured_root = resolve_dataset_root('STRUCTURED3D_ROOT')

# Download only the model weights needed for the selected experiments.
if needs_da360:
    da360_repo = Path(os.environ['DA360_REPO'])
    if not (da360_repo / 'checkpoint.py').exists() or not (da360_repo / 'networks').is_dir():
        if da360_repo.exists():
            raise RuntimeError(
                f'{da360_repo} exists but lacks checkpoint.py + networks, which this DA360 wrapper expects.'
            )
        subprocess.check_call([
            'git', 'clone', '--depth', '1',
            'https://github.com/antigravity-tech/DA360.git', str(da360_repo)
        ])
    da360_weight = Path(os.environ['DA360_WEIGHTS'])
    da360_weight.parent.mkdir(parents=True, exist_ok=True)
    if not da360_weight.exists():
        from huggingface_hub import hf_hub_download
        da360_weight = Path(hf_hub_download(
            repo_id='hljiang/DA360', filename='DA360_large.pth',
            local_dir=str(da360_weight.parent)
        ))
    os.environ['DA360_REPO'] = str(da360_repo)
    os.environ['DA360_WEIGHTS'] = str(da360_weight)
    print('DA360 repo:', da360_repo, '| weights:', da360_weight)

if needs_midas:
    midas_weight = Path(os.environ['MIDAS_WEIGHTS'])
    midas_weight.parent.mkdir(parents=True, exist_ok=True)
    if not midas_weight.exists():
        import urllib.request
        urllib.request.urlretrieve(
            'https://github.com/isl-org/MiDaS/releases/download/v3/dpt_hybrid_384.pt',
            str(midas_weight)
        )
    os.environ['MIDAS_WEIGHTS'] = str(midas_weight)
    print('MiDaS weights:', midas_weight)

# Confirm PanoX EXR reader support. Lokesh's loader falls back to OpenEXR when OpenCV does not.
if needs_panox:
    os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
    import cv2
    first = next((line.split() for line in (panox_root / 'test.txt').read_text().splitlines()
                  if line.strip()), None)
    if first and len(first) > 1:
        exr_path = panox_root / first[1]
        probe = cv2.imread(str(exr_path), cv2.IMREAD_ANYDEPTH | cv2.IMREAD_UNCHANGED)
        if probe is None:
            import OpenEXR
            print('OpenCV EXR decoder unavailable; OpenEXR fallback will read:', exr_path)

print('Setup complete for experiment:', RUN)



## Choose the study

Set RUN below. The script wrappers keep outputs in timestamped folders under followup_experiments/results.
Runs 01, 02, 04 and 05 use GPU 0. Run 02 without a frame limit so every PanoX environment remains represented.
Run 03 training with both devices; its evaluate.py stage is separate and uses the held-out environments.


In [ ]:
# Launch the selected experiment from the setup cell above.
env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0,1' if RUN == '03' or (RUN == 'all' and RUN_FINETUNE_IN_ALL) else '0'
env['RUN_FINETUNE'] = '1' if RUN == 'all' and RUN_FINETUNE_IN_ALL else '0'

if RUN == 'all':
    cmd = ['bash', 'followup_experiments/run_all.sh']
else:
    folders = {
        '01': '01_scale_only_protocol',
        '02': '02_domain_conditioned_gate',
        '03': '03_panda_style_finetuning',
        '04': '04_mono360_20view',
        '05': '05_fov_resolution_sweep',
    }
    if RUN not in folders:
        raise ValueError(f'Unknown RUN={RUN}')
    cmd = ['bash', f'followup_experiments/{folders[RUN]}/run_gpu.sh']
    if LIMIT is not None and RUN in ('01', '04', '05'):
        cmd += ['--limit', str(LIMIT)]

print('Executing:', ' '.join(cmd), '| CUDA_VISIBLE_DEVICES=', env['CUDA_VISIBLE_DEVICES'])
subprocess.run(cmd, cwd=LOKESH_ROOT, env=env, check=True)



## Student evaluation (after RUN='03')

Open the output printed by train.py and copy the resulting best.pt path below. Evaluation defaults to
the held-out environment groups saved next to that checkpoint; it does not evaluate on the training groups.


In [ ]:
CHECKPOINT = ''  # e.g. followup_experiments/03_panda_style_finetuning/outputs/run_YYYYMMDD_HHMMSS/best.pt
if CHECKPOINT:
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = '0'
    subprocess.run(['bash', 'followup_experiments/03_panda_style_finetuning/evaluate_gpu.sh', CHECKPOINT],
                   cwd=LOKESH_ROOT, env=env, check=True)
else:
    print('Set CHECKPOINT after the student-training stage completes.')
